# Feature Construction: Engineering Domain-Informed Variables

Feature construction is the creative process of engineering new variables from existing data to better represent the underlying phenomena. Unlike feature extraction (which algorithmically distills data), feature construction is heavily **domain-driven**.

In this notebook, we will construct features using:
1. **Mathematical Combinations**
2. **Aggregations**
3. **Domain-Specific Logic (RFM)**
4. **Binning & Categorization**
5. **Interaction Features**
6. **Encoding Categorical Data**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt

# Set visual style for any plots
sns.set_theme(style="whitegrid")

### 0. Generating Synthetic Data
We need two tables to properly demonstrate these concepts: a table of **Users** (demographics) and a table of their **Transactions** (behavior).

In [ ]:
np.random.seed(27)
n_users = 511
n_transactions = 3139

# 1. USERS DATAFRAME
users = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'age': np.random.randint(18, 80, n_users),
    'income': np.random.normal(60000, 20000, n_users).astype(int),
    'house_price': np.random.uniform(200000, 1000000, n_users).astype(int),
    'house_sqft': np.random.uniform(1000, 4000, n_users).astype(int),
    'favorite_category': np.random.choice(['Electronics', 'Apparel', 'Home', 'Beauty'], n_users)
})

# Ensure no negative incomes
users['income'] = users['income'].clip(lower=20000)

# 2. TRANSACTIONS DATAFRAME
today = datetime.now()
transactions = pd.DataFrame({
    'transaction_id': range(1, n_transactions + 1),
    'user_id': np.random.choice(range(1, n_users + 1), n_transactions),
    'amount': np.random.exponential(50, n_transactions) + 10, # Skewed spending amounts
    'date': [today - timedelta(days=np.random.randint(0, 365)) for _ in range(n_transactions)]
})

print("Users Table (First 3 rows):")
display(users.head(3))
print("\nTransactions Table (First 3 rows):")
display(transactions.head(3))

### 1. Mathematical Combinations
Applying arithmetic operations to reveal clearer relationships. 

**Why?** A linear model might struggle to link `house_price` and `house_sqft` individually to infer wealth, but `price_per_sqft` is a standard, highly informative real-estate metric.

In [ ]:
# Constructing a new feature: Price per Square Foot
users['price_per_sqft'] = users['house_price'] / users['house_sqft']

display(users[['house_price', 'house_sqft', 'price_per_sqft']].head())

### 2. Aggregations & 3. Domain-Specific Features (RFM)
**RFM (Recency, Frequency, Monetary)** is a powerful domain-specific framework used in retail/marketing to grade customer value. We will construct these features by *aggregating* the raw temporal transactions table.

In [ ]:
# Group transactions by user to construct behavioral features
user_behavior = transactions.groupby('user_id').agg(
    Monetary=('amount', 'sum'),            # Aggregation: Total spent
    Frequency=('transaction_id', 'count'), # Aggregation: Number of purchases
    Last_Purchase=('date', 'max'),         # Find the most recent date
    Avg_Order_Value=('amount', 'mean')     # Aggregation: Average spent per order
).reset_index()

# Domain Logic: Calculate 'Recency' in days (How long since they last bought?)
user_behavior['Recency_Days'] = (today - user_behavior['Last_Purchase']).dt.days
user_behavior = user_behavior.drop(columns=['Last_Purchase']) # Drop raw datetime object

print("Domain-Engineered RFM + Aggregation Features:")
display(user_behavior.head())

# Merge these newly constructed features back into our main users table
users = users.merge(user_behavior, on='user_id', how='left').fillna(0)

### 4. Binning & Categorization
Transforming continuous variables into discrete buckets to handle nonlinearities. 

**Why?** The behavioral difference between a 20-year-old and a 25-year-old is minimal, but the difference between a 25-year-old and a 60-year-old is massive. Binning captures these generational steps.

In [ ]:
# Define bin edges and labels for Age cohorts
bins = [0, 26, 42, 58, 100]
labels = ['Gen_Z', 'Millennial', 'Gen_X', 'Boomer']

# Construct categorical feature using pandas cut
users['generation_cohort'] = pd.cut(users['age'], bins=bins, labels=labels)

display(users[['age', 'generation_cohort']].head(6))

### 5. Interaction Features
Modeling dependencies between features. Sometimes the *combination* of two features holds the true predictive power. 

**Example:** `Wealth_Engagement_Index` = `Income` $\times$ `Frequency` (Captures the synergistic effect of highly engaged users who also have high purchasing power).

In [ ]:
# Create an interaction term
users['wealth_engagement_index'] = users['income'] * users['Frequency']

display(users[['income', 'Frequency', 'wealth_engagement_index']].head())

### 6. Encoding Categorical Features
Machine learning models require numeric inputs. We use **One-Hot Encoding** to transform non-numeric categories (`favorite_category`) into distinct, machine-readable binary columns.

In [ ]:
# One-Hot Encode 'favorite_category'
users_encoded = pd.get_dummies(users, columns=['favorite_category'], prefix='fav_cat', dtype=int)

print("Final Engineered Dataset:")
display(users_encoded.head())
print(f"\nOriginal Dataset Columns: 6")
print(f"Engineered Dataset Columns: {users_encoded.shape[1]}")

### Final Summary
Through domain knowledge, we transformed basic demographic and transactional records into rich, highly predictive variables (`price_per_sqft`, `Recency_Days`, `generation_cohort`, and `wealth_engagement_index`). 

This constructed dataset is now vastly superior for training a machine learning model compared to the raw inputs we started with!